In [ ]:
import pandas as pd

# --- CONSTANTES ---
# Definición de rutas y columnas para facilitar el mantenimiento del código
RUTA_TRAMOS = "../data/raw/Tramos_Iti.csv"
RUTA_UBICACIONES = "../data/raw/Tramos_Ubi.csv"
RUTA_DATOS_ENERO = "../data/raw/2026_01.csv"

# Listas de columnas a procesar
COLS_ENERO = ['idTram', 'data', 'tempsActual', 'tempsPrevist']
COLS_TRAMOS = ['idTram', 'Descripcion', 'Tram']

# --- CARGA DE DATOS ---
tramos_df = pd.read_csv(RUTA_TRAMOS)
ubicaciones_df = pd.read_csv(RUTA_UBICACIONES)
enero_df = pd.read_csv(RUTA_DATOS_ENERO)

# Visualización inicial para verificar integridad de carga
display(tramos_df.head(5))
display(ubicaciones_df.head(5))
display(enero_df.head(5))

# --- TRANSFORMACIÓN Y UNIÓN ---

# Decisión de diseño: Usar inner join para descartar registros de enero 
# que no tengan una descripción técnica asociada en el catálogo de tramos.
primer_union_df = pd.merge(
    tramos_df[COLS_TRAMOS],
    enero_df[COLS_ENERO],
    on='idTram',
    how='inner'
)

# Decisión de diseño: Cruzar con ubicaciones mediante 'Descripcion' 
# para obtener coordenadas geográficas necesarias en la visualización de mapas.
union_final_df = pd.merge(
    primer_union_df,
    ubicaciones_df,
    on='Descripcion',
    how='inner'
)

# Limpieza de columnas duplicadas tras el merge
# Si existen Tram_x o Tram_y, las consolidamos en una sola columna 'Tram'
if 'Tram_x' in union_final_df.columns:
    union_final_df['Tram'] = union_final_df['Tram_x']
    union_final_df.drop(columns=['Tram_x', 'Tram_y'], errors='ignore', inplace=True)
# Esto para poder ingresar Tram en el "group by"

# 1. Asegurar que 'data' sea texto y convertir a fecha
union_final_df['data'] = union_final_df['data'].astype(str)
union_final_df['fecha_limpia'] = pd.to_datetime(union_final_df['data'], format='%Y%m%d%H%M%S')

# 2. Establecer el índice
df_reducido = union_final_df.set_index('fecha_limpia')

# 3. REALIZAR LA REDUCCIÓN (Ahora 'Tram' ya existe sin el _x)
# Agrupamos por idTram, Tram y Descripcion para mantener la estructura de la calle
df_promediado = df_reducido.groupby(['idTram', 'Tram', 'Descripcion']).resample('h')[['tempsActual']].mean()

# 4. Resetear el índice para que todo vuelva a ser columnas normales
df_final_procesado = df_promediado.reset_index()

# 5. Ver el resultado
print("¡Reducción completada con éxito!")
print(df_promediado.head())


# --- EXPORTACIÓN ---
# Decisión de diseño: En CSV Para poder visualizar los datos antes de convertirlos en xml.
df_promediado.to_csv("../data/processed/ARCHIVO_SALIDA_PROMEDIO.csv", index=True, encoding='utf-8')

,idTram,Tram,Descripcion
0,1,0,Aragó de Meridiana a Pau Claris
1,1,271,Aragó (Meridiana a Cartagena)
2,1,272,Aragó (Cartagena a Diagonal)
3,1,273,Aragó (Diagonal a Passeig de Sant Joan)
4,1,274,Aragó (Passeig de Sant Joan a Pau Claris)


,Tram,Descripcion,Coordenades
0,1,Diagonal (Ronda de Dalt a Doctor Marañón),"2.11203535639414,41.3841912394771,2.1015028628..."
1,2,Diagonal (Doctor Marañón a Ronda de Dalt),"2.111944376806616,41.38446666680338,2.10159408..."
2,3,Diagonal (Doctor Marañón a Pl. Pius XII),"2.112093343037027,41.38422850920645,2.12264979..."
3,4,Diagonal (Pl. Pius XII a Doctor Marañón),"2.122592049318304,41.38719094189204,2.11196902..."
4,5,Diagonal (Pl. Pius XII a Pl. Maria Cristina),"2.122657659295115,41.38694195794678,2.12755961..."


,idTram,infoDisponible,data,tempsActual,tempsPrevist,tempsRecorregutFutur,factorReferenciaActual,tendencia
0,1,1,20260101001556,300,300,1,1,1
1,2,1,20260101001556,0,0,-1,-1,-1
2,3,1,20260101001556,304,306,1,1,1
3,4,1,20260101001556,325,325,1,1,1
4,5,1,20260101001556,174,174,1,1,1


¡Reducción completada con éxito!
                                                               tempsActual
idTram Tram Descripcion                   fecha_limpia                    
1      271  Aragó (Meridiana a Cartagena) 2026-01-01 00:00:00   301.750000
                                          2026-01-01 01:00:00   303.833333
                                          2026-01-01 02:00:00   302.400000
                                          2026-01-01 03:00:00   241.200000
                                          2026-01-01 04:00:00   301.000000
